### old version but working. very slow

In [1]:
import torch
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModel, pipeline
from transformers.pipelines.pt_utils import KeyDataset
from datasets import Dataset
from tqdm import tqdm

from torch.utils.data import TensorDataset, DataLoader

from mhcicleave.lightning_model import CleavageLightningModule
from mhcicleave.models import MLP

import lightning.pytorch as pl

2025-09-23 11:22:06.566602: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-09-23 11:22:06.843771: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1758637326.995740  439885 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1758637327.046588  439885 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1758637327.261828  439885 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [2]:
def parse_fasta(filepath):
    """
    Parses a FASTA file and returns a dictionary where keys are sequence IDs
    and values are the corresponding sequences.

    Args:
        filepath (str): The path to the FASTA file.

    Returns:
        dict: A dictionary of sequence IDs and sequences.
    """
    sequences = {}
    current_id = None
    current_sequence = []

    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:  # Skip empty lines
                continue
            if line.startswith('>'):
                if current_id is not None:
                    sequences[current_id] = "".join(current_sequence)
                current_id = line[1:].split()[0]  # Take the first word as ID
                current_sequence = []
            else:
                current_sequence.append(line)

        # Add the last sequence after the loop finishes
        if current_id is not None:
            sequences[current_id] = "".join(current_sequence)

    return sequences

def format_prediction(predictions, threshold=0.5):

    predictions[-1] = 0 # last position by def is not a cleavage site
    out_preds = [round(p, 4) for p in predictions]
    positions = range(1, len(predictions)+1)
    residues = list(protseq)
    cleave = [p > threshold for p in predictions]
    prot_list = [protid] * len(positions)
    out_zip = zip(positions, residues, out_preds, cleave, prot_list)
    out = [i for i in out_zip]

    pd_out = pd.DataFrame(out, columns=["resnum","res","pred","cleave","seqname"])

    return pd_out

In [3]:
def get_peptide_window(sequence, starting_position, ending_position, upstream=6,
                       downstream=6, c_terminal=True, pad_str = 'X'):
    """
    returns the window of AA's around the C-term of an peptide, given defined
    upstream and downstream window sizes and a row from a pandas df with
    ending position and full origin sequence of the peptide.
    :param pd_entry: pandas entry with (at min.) ending_position and sequence
    :param sequence: protein sequence from which to extract peptide window
    :param starting_position: N terminal starting position for peptide
                              (1 based, inclusive)
    :param ending_position: C terminal ending position for peptide (1 based,
                             exclusive)
    :param upstream: number of upstream AA's to return
    :param downstream: number of downstream AA's to return
    :param c_terminal: whether c or n terminal cleavage site is to be used for
    window midpoint
    :return: full window of AA's including the cleavage site
    """
    # set cleavage site index based on function flag
    seq_len = len(sequence)
    if c_terminal:
        cleave_index = int(ending_position) - 1
    else:
        cleave_index = int(starting_position) - 1

    if cleave_index < 0 or (cleave_index - 1) >= seq_len:
        return None

    # if upstream window does not hit boundary
    if (cleave_index - upstream) >= 0:
        upstream_seq = sequence[(cleave_index - upstream):cleave_index]
    # if peptide boundary is within upstream window
    else:
        # retrieve relevant sequence
        tmp_seq = sequence[0:cleave_index]
        # add empty AA's prior to seq start
        upstream_seq = abs(cleave_index - upstream) * pad_str + tmp_seq
    # repeat above with downstream window
    if (cleave_index + 1 + downstream) < seq_len:
        downstream_seq = sequence[(cleave_index + 1):(cleave_index +
                                                      downstream + 1)]
    else:
        # handles issue where cleavage site was end of protein and
        # cleave_index + 1 was beyond sequence bounds
        if cleave_index == (seq_len - 1):
            downstream_seq = downstream * pad_str
        else:
            tmp_seq = sequence[(cleave_index + 1):seq_len]
            downstream_seq = tmp_seq + (downstream + 1 + cleave_index -
                                        seq_len) * pad_str
    # return up/down stream windows + cleavage site
    return upstream_seq + sequence[cleave_index] + downstream_seq


def create_windows_from_protein(protein_seq, **kwargs):
    """
    wrapper for get_peptide_window(). takes in a protein sequence and returns
    a vector of k-merized windows.
    :param protein_seq: protein sequence
    :return: vector of protein windows
    """
    # NOTE: last AA not made into window since c-terminal would be cleavage pos
    protein_windows = []
    for pos in range(len(protein_seq)):
        start_pos = pos + 1
        end_pos = pos + 1
        
        tmp_window = get_peptide_window(protein_seq,
                                            starting_position=start_pos,
                                            ending_position=end_pos,
                                            **kwargs)
        protein_windows.append(tmp_window)

    return protein_windows


In [4]:
class Embedder:

    def __init__(self, name: str = "facebook/esm2_t33_650M_UR50D", device = 'cpu'):
        
        ALLOWED_MODELS = ['Rostlab/prot_bert', 'Rostlab/prot_t5_xl_uniref50', 'facebook/esm2_t33_650M_UR50D', 'onehot']
        assert name in ALLOWED_MODELS, f"This function is only built for {ALLOWED_MODELS}, get {name}"

        self.device = device
        print('Initializing protein sequence embedder ... ', end='')
        #logging.info(f"Loading ESM-2 model ({name})...")
    
        if name == 'Rostlab/prot_bert':
            self.featureLen = 1024
            self.tokenizer = BertTokenizer.from_pretrained(name, do_lower_case=False)
            self.model = BertModel.from_pretrained(name).to(self.device).eval()
            self.encode = self.encode_prot

        elif name == 'Rostlab/prot_t5_xl_uniref50':
            self.featureLen = 1024
            self.tokenizer = T5Tokenizer.from_pretrained(name, do_lower_case=False )
            self.model = T5EncoderModel.from_pretrained(name).to(self.device).eval()
            self.encode = self.encode_prot

        elif name == 'facebook/esm2_t33_650M_UR50D':
            self.featureLen = 1280
            self.tokenizer = AutoTokenizer.from_pretrained(name)
            self.model =  AutoModel.from_pretrained(name).to(self.device).eval()
            self.encode = self.encode_esm
        
        elif name == 'onehot':
            self.featureLen = 21
            self.encode = self.encode_onehot

    def encode_esm(self, sequences, emb_type='concat'):

        pipe = pipeline('feature-extraction', model=self.model, tokenizer=self.tokenizer, framework='pt')
        #key_dataset = KeyDataset(dataset, sequence_col)
        
        embeddings_list = []
        for out in tqdm(pipe(sequences, batch_size=16, return_tensors=True,num_workers=10,shuffle=False),
                        total=len(sequences), desc='Generating Embeddings'):
            embeddings = out[:, 1:-1, :]
            if emb_type == 'concat':
                emb = torch.flatten(embeddings, start_dim=1)
            embeddings_list.append(emb)
        embeddings_result = torch.stack(embeddings_list)
        #del self.model
        
        return embeddings_result




In [5]:
# args
device = 'cuda'
fasta_file = "vax_seqs_test.fasta"
checkpoint = "../models/best_model-epoch=13-val_loss=0.32.ckpt"

In [6]:
%%time

device = torch.device(device if torch.cuda.is_available() else 'cpu')


# Modelo MLP:
seq_len = 10 # largo cleavage window
esm2_dim = 1280
input_size_mlp = seq_len * esm2_dim

model_params_mlp = {"input_size": input_size_mlp, "hidden_linear1": 256, "hidden_linear2": 128, "out_neurons": 1, "dropout": 0.5}
model_network = MLP(**model_params_mlp)


print(f"Loading cleavage model {checkpoint} ...")
try:
    model = CleavageLightningModule.load_from_checkpoint("../models/best_model-epoch=13-val_loss=0.32.ckpt",model=model_network)
except Exception as e:
    print(f"Error loading model. Make sure the class 'GenericClassifier' is defined and imported correctly.")
    raise e

trainer = pl.Trainer()
embedder = Embedder('facebook/esm2_t33_650M_UR50D', device)

prot_results = []
protein_list = parse_fasta(fasta_file)
for protid, protseq in protein_list.items():

    upstream = 5
    downstream = 4
    protein_windows = create_windows_from_protein(protseq, upstream=upstream, downstream=downstream)
    
    seq_embeddings = embedder.encode_esm(protein_windows)

    dataset = TensorDataset(seq_embeddings)
    dataloader = DataLoader(dataset, batch_size=8, shuffle=False, num_workers=10)

    trainer = pl.Trainer()
    predictions = trainer.predict(model, dataloader)
    predictions = torch.cat([batch.flatten() for batch in predictions]).numpy()

    pd_out = format_prediction(predictions)
    prot_results.append(pd_out)

Loading cleavage model best_model-epoch=11-val_BinaryAUROC=0.941.ckpt ...


/home/vilion/miniforge3/envs/pLM/lib/python3.12/site-packages/torchmetrics/utilities/prints.py:43: FutureWarning: The default value for `maximize` will be changed from `True` to `None` in v1.7.0 of TorchMetrics,will automatically infer the value based on the `higher_is_better` attribute of the metric (if such attribute exists) or raise an error if it does not. If you are explicitly setting the `maximize` argument to either `True` or `False` already, you can ignore this warning.
  warnings.warn(*args, **kwargs)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


Initializing protein sequence embedder ... 

Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t33_650M_UR50D and are newly initialized: ['esm.pooler.dense.bias', 'esm.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use cuda:0
Generating Embeddings: 100%|█████████████████████████████████████████████████████████████████████████████████████████████| 18/18 [00:00<00:00, 35164.17it/s]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA GeForce RTX 4070 Laptop GPU') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEV

Predicting: |                                                                                                 …

Device set to use cuda:0
Generating Embeddings: 100%|█████████████████████████████████████████████████████████████████████████████████████████████| 18/18 [00:00<00:00, 81884.46it/s]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


... (194 outputs left out)

Predicting: |                                                                                                 …

Device set to use cuda:0
Generating Embeddings: 100%|█████████████████████████████████████████████████████████████████████████████████████████████| 18/18 [00:00<00:00, 43868.37it/s]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting: |                                                                                                 …

CPU times: user 56.3 s, sys: 48.1 s, total: 1min 44s
Wall time: 4min 16s


In [28]:
seq_scores = []
for protdf in prot_results:
    segments= [('CALNN', 5), ('Linker1', 4), ('Epitope1', 9)]
    seqname = protdf.seqname.unique()[0]

    pred = protdf.pred
    vax_process_score = calculate_cleavage_score(pred, segments)
    seq_scores.append((seqname,vax_process_score))


In [30]:
protdf

,resnum,res,pred,cleave,seqname
0,1,C,0.1461,False,seq100_LPQL
1,2,A,0.0694,False,seq100_LPQL
2,3,L,0.3556,False,seq100_LPQL
3,4,N,0.0159,False,seq100_LPQL
4,5,N,0.0117,False,seq100_LPQL
5,6,L,0.1867,False,seq100_LPQL
6,7,P,0.0364,False,seq100_LPQL
7,8,Q,0.0155,False,seq100_LPQL
8,9,L,0.3505,False,seq100_LPQL
9,10,D,0.0568,False,seq100_LPQL


In [33]:
sorted(seq_scores, key=lambda x: x[1],reverse=True)

[('seq77_EEHW', -23.221109917292775),
 ('seq93_AWNR', -23.261142268718615),
 ('seq69_QWEE', -23.31742781867127),
 ('seq88_QKYN', -23.367119927499726),
 ('seq0_EAAY', -23.380881134816768),
 ('seq50_NAYT', -23.411906221407396),
 ('seq43_GEWN', -23.425847288031964),
 ('seq32_AIAW', -23.434902921812295),
 ('seq87_IVYH', -23.476460946677758),
 ('seq18_HHGW', -23.497683450464297),
 ('seq34_VWDM', -23.498117961212408),
 ('seq9_NYKN', -23.519602985700637),
 ('seq4_AFQN', -23.541103519476664),
 ('seq27_HQEL', -23.58287078567007),
 ('seq24_TLDK', -23.619062305923542),
 ('seq46_RQYD', -23.6319984433793),
 ('seq71_RFQH', -23.647652536516496),
 ('seq58_EYDF', -23.664322405700716),
 ('seq53_KYHS', -23.66599625839627),
 ('seq80_KPEW', -23.67584207111153),
 ('seq81_VYTV', -23.680322659482563),
 ('seq72_MWVD', -23.68853927963409),
 ('seq28_LYVH', -23.730850529993646),
 ('seq14_HWQP', -23.736857745710182),
 ('seq16_YQNR', -23.793126644952746),
 ('seq78_TVWF', -23.80214388619298),
 ('seq82_ARIW', -23.806

In [8]:
import pandas as pd
import numpy as np
from typing import List, Tuple

# A small constant to prevent log(0) errors
EPSILON = 1e-9

def calculate_cleavage_score(
    cleavage_preds: np.ndarray,
    segments: List[Tuple[str, int]],
    core_len: int = 9
) -> float:
    """
    Calculates a specialized cleavage score for a generic vaccine construct.

    Args:
        cleavage_preds: A NumPy array of cleavage probabilities for the full sequence.
        segments: A list of tuples, where each tuple is (segment_name, segment_length).
                  Example: [('Leader', 5), ('Linker1', 4), ('Epitope1', 15)]
        core_len: The length of the core binding motif to protect within 'Epitope' segments.

    Returns:
        The final composite cleavage score.
    """
    score_protection = 0.0
    score_processing = 0.0
    current_pos = 0

    for seg_name, seg_len in segments:
        if seg_len == 0:
            continue
        
        # Isolate the predictions for the current segment
        seg_preds = cleavage_preds[current_pos : current_pos + seg_len]

        # Apply scoring logic based on segment name
        if 'epitope' in seg_name.lower():
            # --- Epitope Segment: Protect the core, reward flank cleavage ---
            core_offset = (seg_len - core_len) // 2
            core_start_local = core_offset
            core_end_local = core_start_local + core_len
            
            # Penalize cleavage within the core motif
            core_preds = seg_preds[core_start_local : core_end_local - 1]
            score_protection += np.log(1 - core_preds + EPSILON).sum()
            
            # Reward cleavage in the flanking regions
            flank_preds = np.concatenate([
                seg_preds[0 : core_start_local],
                seg_preds[core_end_local - 1 : seg_len]
            ])
            if flank_preds.size > 0:
                score_processing += np.log(flank_preds.max() + EPSILON)
        else:
            # --- Other Segments (Linker, Leader, etc.): Reward cleavage ---
            if seg_preds.size > 0:
                score_processing += np.log(seg_preds.max() + EPSILON)

        # Move to the start of the next segment
        current_pos += seg_len
        
    return score_protection + score_processing


In [11]:
segments_vax = [('CALNN', 5), ('Linker1', 4), ('Epitope1', 9)]
calculate_cleavage_score(predictions,segments_vax)

-23.64985794967378

In [15]:
# --- CASO 1: Constructo [Linker-Epitope] ---
print("--- Caso de prueba 1: [L-E] ---")

# Definir la estructura y generar datos de clivaje simulados
segments_le = [('Linker1', 4), ('Epitope1', 15)]
full_len_le = sum(length for name, length in segments_le)
preds_le = np.random.rand(full_len_le)

# Calcular la puntuación
score1 = calculate_cleavage_score(preds_le, segments_le)
print(f"Estructura: {segments_le}")
print(f"Puntuación de clivaje: {score1:.4f}\n")


# --- CASO 2: Constructo [Leader-Linker-Epitope] (con 'CALNN' por defecto) ---
print("--- Caso de prueba 2: [Leader-L-E] ---")

# Definir la estructura y generar datos de clivaje simulados
segments_lle = [('CALNN', 5), ('Linker1', 4), ('Epitope1', 15)]
full_len_lle = sum(length for name, length in segments_lle)
preds_lle = np.random.rand(full_len_lle)

# Simular un buen clivaje en el linker para ver el efecto en la puntuación
preds_lle[6] = 0.95 # Índice 6 está dentro del Linker1

# Calcular la puntuación
score2 = calculate_cleavage_score(preds_lle, segments_lle)
print(f"Estructura: {segments_lle}")
print(f"Puntuación de clivaje: {score2:.4f}")

--- Caso de prueba 1: [L-E] ---
Estructura: [('Linker1', 4), ('Epitope1', 15)]
Puntuación de clivaje: -12.8932

--- Caso de prueba 2: [Leader-L-E] ---
Estructura: [('CALNN', 5), ('Linker1', 4), ('Epitope1', 15)]
Puntuación de clivaje: -6.7735
